# Evaluate SkewD

In [11]:
folder_names = ["AN", "AN-s", "LS", "LS-s", "MN-U", "SIM", "SIMc", "SIMln", "SIMG", "Tuebingen", "Cha", "Net", "Multi"]

folders_skew = [
                "pure_ANs_SN_med", "pure_ANs_SN_strong", "pure_ANs_AGN_med", "pure_ANs_AGN_strong", "pure_ANs_AGN_extreme", "pure_LSs_SN_med", "pure_LSs_SN_strong", "pure_LSs_AGN_med", "pure_LSs_AGN_strong", "pure_LSs_AGN_extreme"]

In [12]:
import pickle 
from pathlib import Path
import numpy as np
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import csv


In [13]:
import pandas as pd

def load_tuebingen_weights():
    weights_df = pd.read_csv("results/Tuebingen_weights.csv", sep=",")  # or sep="," if it's CSV
    weights = dict(zip(weights_df["PairID"], weights_df["Weight"]))
    return weights

def calc_acc_weighted(results_dict, strat, weights):
    weighted_correct = 0
    for j in results_dict:
        if strat == "IT":
            weighted_correct += weights[j] * int(results_dict[j]["correct_IT"])
        else:
            weighted_correct += weights[j] * int(results_dict[j]["correct_LL"])
    return weighted_correct / np.sum(np.array([j for j in weights.values()]))

def calc_AUDRC_weighted(results_dict, strat, weights):
    if strat == "IT":
        result_list = [
            (abs(results_dict[j]["indep_score"]), int(results_dict[j]["correct_IT"]), weights[j])
            for j in results_dict
        ]
    else:
        result_list = [
            (abs(results_dict[j]["lik_score"]), int(results_dict[j]["correct_LL"]), weights[j])
            for j in results_dict
        ]
    # Sort by abs score
    result_list.sort(reverse=True, key=lambda x: x[0])

    weights_sorted = np.array([r[2] for r in result_list])
    correct_sorted = np.array([r[1] for r in result_list])
    correct_weighted = weights_sorted * correct_sorted

    cum_cw = np.cumsum(correct_weighted)
    cum_w = np.cumsum(weights_sorted)

    cp = cum_cw/cum_w

    audrc = np.sum(weights_sorted * cp)/np.sum(weights_sorted)
    return(audrc)

In [14]:
def read_results(folder):
    if folder in ["Multi", "Net", "Cha"]:
        n_pairs = 300
    else:
        n_pairs = 100
    iterover = range(1, n_pairs + 1)
    if folder == "Tuebingen": # 108 pairs but [47, 52, 53, 54, 55, 70, 71, 105, 107] excluded
        iterover = [pid for pid in iterover if pid not in [47, 52, 53, 54, 55, 70, 71, 105, 107]]    

    folder_name = folder

    if folder == "MN-U":
        folder_name = "MNU"
    elif folder == "Cha":
        folder_name = "CE-Cha"
    elif folder == "Net":
        folder_name = "CE-Net"
    elif folder == "Multi":
        folder_name = "CE-Multi"

    if folder in ["Multi", "Net", "Cha"]:
        superfolder = "Dataverse_pairs_results"
    elif folder in ["SIM", "SIMc", "SIMln", "SIMG"]:
        superfolder = "Benchmark_simulated_results"
    elif folder in ["AN", "AN-s", "LS", "LS-s", "MN-U"]:
        superfolder = "ANLSMN_results"
    else:
        superfolder = "skew_results"
    res = {}
    for j in iterover:
        if folder == "Tuebingen":
            save_file = Path("results") / "Tuebingen_results" / f"result_{j}.pkl"
        else:
            save_file = Path("results") / superfolder / folder_name / f"result_{j}.pkl"
        with open(save_file, 'rb') as file:
            res[j] = pickle.load(file)
    return res


def calc_acc(results_dict, strat="IT"):
    results = results_dict
    n = len(results.keys())
    n_correct = 0
    if strat == "IT":
        for j in results.keys():
            n_correct += results[j]["correct_IT"]    
    elif strat == "LL":
        for j in results.keys():
            n_correct += results[j]["correct_LL"]
    else:
        raise ValueError("strat needs to be LL or IT")
    return n_correct/n

def calc_AUDRC(results_dict, strat="IT"):
    correct_key = f"correct_{strat}"
    score_key = "indep_score" if strat == "IT" else "lik_score"

    sorted_items = sorted(
        results_dict.items(),
        key=lambda item: abs(item[1][score_key]), # item[1] is the value of the (key, value pair) 
        reverse=True
    )

    correct_predictions = [item[1][correct_key] for item in sorted_items]
    cumulative_correct = np.cumsum(correct_predictions)
    decision_recall = cumulative_correct / (np.arange(1, len(correct_predictions) + 1))

    auc = decision_recall.mean()
    return auc

In [15]:
def write_csv(folders_nonskew, folders_skew):
    acc_rows = [["Method"] + folders_nonskew + folders_skew]
    audrc_rows = [["Method"] + folders_nonskew + folders_skew]

    acc_LL = ["SkewD-LL"]
    acc_IT = ["SkewD-IT"]
    audrc_LL = ["SkewD-LL"]
    audrc_IT = ["SkewD-IT"]

    for folder in folders_nonskew:            
        results = read_results(folder)
        if folder == "Tuebingen":
            weights = load_tuebingen_weights()
            acc_LL.append(calc_acc_weighted(results, strat="LL", weights=weights))
            audrc_LL.append(calc_AUDRC_weighted(results, strat="LL", weights=weights))

            acc_IT.append(calc_acc_weighted(results, strat="IT", weights=weights))
            audrc_IT.append(calc_AUDRC_weighted(results, strat="IT", weights=weights))
        else:
            acc_LL.append(calc_acc(results, strat="LL"))
            audrc_LL.append(calc_AUDRC(results, strat="LL"))

            acc_IT.append(calc_acc(results, strat="IT"))
            audrc_IT.append(calc_AUDRC(results, strat="IT"))

    for folder in folders_skew:
        results = read_results(folder)

        acc_LL.append(calc_acc(results, strat="LL"))
        audrc_LL.append(calc_AUDRC(results, strat="LL"))

        acc_IT.append(calc_acc(results, strat="IT"))
        audrc_IT.append(calc_AUDRC(results, strat="IT"))

    acc_rows += [acc_LL, acc_IT]
    audrc_rows += [audrc_LL, audrc_IT]

    with open("results/skewd_acc.csv", "w", newline="") as f_acc, open("results/skewd_audrc.csv", "w", newline="") as f_audrc:
        writer_acc = csv.writer(f_acc)
        writer_audrc = csv.writer(f_audrc)

        writer_acc.writerows(acc_rows)
        writer_audrc.writerows(audrc_rows)


In [16]:
write_csv(folder_names, folders_skew)

# Now calculate acc, AUDRC for merged pure datasets:

In [17]:
names = ["ANs_med", "ANs_strong", "LSs_med", "LSs_strong"]

In [18]:
def read_csw_merged(name):
    parts = name.split("_")
    res_AGN = read_results(f"pure_{parts[0]}_AGN_{parts[1]}")
    res_SN = read_results(f"pure_{parts[0]}_SN_{parts[1]}")
    joint_dict = {}
    for j in range(1, 201):
        if j < 101:
            k = j
            joint_dict[j] = res_SN[k]
        else:
            k = j - 100
            joint_dict[j] = res_AGN[k]
    return joint_dict

In [19]:
def write_csv_merge(names):
    acc_rows = [["Method"] + names]
    audrc_rows = [["Method"] + names]

    acc_LL = ["SkewD-LL"]
    acc_IT = ["SkewD-IT"]
    audrc_LL = ["SkewD-LL"]
    audrc_IT = ["SkewD-IT"]

    for folder in names:            
        results = read_csw_merged(folder)

        acc_LL.append(calc_acc(results, strat="LL"))
        audrc_LL.append(calc_AUDRC(results, strat="LL"))

        acc_IT.append(calc_acc(results, strat="IT"))
        audrc_IT.append(calc_AUDRC(results, strat="IT"))

    acc_rows += [acc_LL, acc_IT]
    audrc_rows += [audrc_LL, audrc_IT]

    with open("results/skewd_acc_merge.csv", "w", newline="") as f_acc, open("results/skewd_audrc_merge.csv", "w", newline="") as f_audrc:
        writer_acc = csv.writer(f_acc)
        writer_audrc = csv.writer(f_audrc)

        writer_acc.writerows(acc_rows)
        writer_audrc.writerows(audrc_rows)


In [20]:
write_csv_merge(names)